# Snowflake Iceberg Interop Explorer — Complete Story (28 Features)

**Import into Snowsight:** Projects → Notebooks → Import → select this file

Uses `get_active_session()` — zero auth setup, works out of the box.

| # | Feature | Category |
|---|---------|----------|
| 1 | Open Iceberg REST Access | Core |
| 2 | Horizon Catalog (Polaris-based) | Core |
| 3 | Single Endpoint | Core |
| 4 | External Engine Read + Write | Interop |
| 5 | Snowflake Security Model | Governance |
| 6 | Credential Vending | Governance |
| 7 | Governed Multi-Engine Access | Governance |
| 8 | Policy Enforcement on Iceberg | Governance |
| 9 | Supported External Engines | Reference |
| 10 | Snowflake Storage for Iceberg | Storage |
| 11 | Catalog Federation / Catalog-Linked DBs | Catalog |
| 12 | Iceberg Time Travel | Data Management |
| 13 | Automated Table Maintenance | Operations |
| 14 | Schema Evolution | Operations |
| 15 | Competitive Positioning | Reference |
| 16 | Snowpipe Streaming → Iceberg | Ingestion |
| 17 | Dynamic Tables as Iceberg | Pipelines |
| 18 | Partitioning + Performance Tuning | Performance |
| 19 | Secure Data Sharing for Iceberg | Sharing |
| 20 | Snowpark on Iceberg | Native Python |
| 21 | Object Tags + Data Classification | Governance |
| 22 | Unity Catalog ↔ Horizon Catalog | Multi-Cloud |
| 23 | PrivateLink for Catalog Integrations | Security |
| 24 | Iceberg v3 Features | Format |
| 25 | WIF / OIDC Authentication for External Engines | Security |
| 26 | Supported External Catalogs | Reference |
| 27 | Microsoft OneLake REST Catalog | Catalog |
| 28 | Automatic Table Discovery + Remote Catalog Sync | Catalog |


## 0. Setup — Configure your account details, create demo database + Iceberg table

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, json

session = get_active_session()

# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
EXTERNAL_VOLUME = 'iceberg_demo_ext_vol'
DATABASE        = 'horizon_demo_db'
SCHEMA          = 'public'
WAREHOUSE       = 'OPENFLOW_INGEST_WAREHOUSE'
# ─────────────────────────────────────────────────────────────────────────────

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'CREATE DATABASE IF NOT EXISTS {DATABASE}').collect()
session.sql(f'CREATE SCHEMA IF NOT EXISTS {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {SCHEMA}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()

# Get Horizon endpoint — with fallback if function not available on this account
try:
    HORIZON_URI = session.sql(
        "SELECT SYSTEM$GET_ICEBERG_REST_CATALOG_ENDPOINT() AS e"
    ).to_pandas()['E'].iloc[0]
except Exception:
    account_id = session.sql("SELECT LOWER(CURRENT_ACCOUNT()) AS a").to_pandas()['A'].iloc[0]
    HORIZON_URI = f"https://{account_id}.snowflakecomputing.com/polaris/api/catalog"

print(f"Account   : {session.get_current_account()}")
print(f"Role      : {session.get_current_role()}")
print(f"Warehouse : {session.get_current_warehouse()}")
print(f"Database  : {DATABASE}.{SCHEMA}")
print(f"Horizon   : {HORIZON_URI}")

In [ ]:
session.sql(f'''
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions (
    transaction_id   STRING,
    customer_id      STRING,
    amount           DECIMAL(12,2),
    currency         STRING,
    transaction_ts   TIMESTAMP_NTZ(6),
    status           STRING,
    region           STRING
)
    CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}'
    BASE_LOCATION='horizon_demo/transactions/'
''').collect()

session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.transactions VALUES
    ('txn-001','cust-A',1250.00,'USD','2024-01-15 09:30:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west'),
    ('txn-002','cust-B', 320.50,'USD','2024-01-15 10:15:00'::TIMESTAMP_NTZ(6),'PENDING',  'us-east'),
    ('txn-003','cust-A', 875.00,'EUR','2024-01-15 11:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-004','cust-C',4200.00,'USD','2024-01-15 12:45:00'::TIMESTAMP_NTZ(6),'FAILED',   'us-west'),
    ('txn-005','cust-B', 650.75,'GBP','2024-01-15 13:30:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-006','cust-D', 980.00,'USD','2024-01-16 08:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-east'),
    ('txn-007','cust-E', 450.00,'EUR','2024-01-16 09:15:00'::TIMESTAMP_NTZ(6),'PENDING',  'eu-west'),
    ('txn-008','cust-F',1100.00,'USD','2024-01-17 10:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west'),
    ('txn-009','cust-G', 220.50,'GBP','2024-01-17 11:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-010','cust-A', 340.00,'USD','2024-01-18 08:30:00'::TIMESTAMP_NTZ(6),'FAILED',   'us-east')
""").collect()

session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()

session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.transactions VALUES
    ('txn-001','cust-A',1250.00,'USD','2024-01-15 09:30:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west'),
    ('txn-002','cust-B', 320.50,'USD','2024-01-15 10:15:00'::TIMESTAMP_NTZ(6),'PENDING',  'us-east'),
    ('txn-003','cust-A', 875.00,'EUR','2024-01-15 11:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-004','cust-C',4200.00,'USD','2024-01-15 12:45:00'::TIMESTAMP_NTZ(6),'FAILED',   'us-west'),
    ('txn-005','cust-B', 650.75,'GBP','2024-01-15 13:30:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-006','cust-D', 980.00,'USD','2024-01-16 08:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-east'),
    ('txn-007','cust-E', 450.00,'EUR','2024-01-16 09:15:00'::TIMESTAMP_NTZ(6),'PENDING',  'eu-west'),
    ('txn-008','cust-F',1100.00,'USD','2024-01-17 10:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west'),
    ('txn-009','cust-G', 220.50,'GBP','2024-01-17 11:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-010','cust-A', 340.00,'USD','2024-01-18 08:30:00'::TIMESTAMP_NTZ(6),'FAILED',   'us-east')
""").collect()

session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()

---
## Feature 1 — Open Iceberg REST Access
Horizon Catalog exposes every Iceberg table via the **Apache Iceberg REST Catalog API**. Any engine that speaks the open REST protocol can query Snowflake tables — no proprietary connector needed.

In [ ]:
print(f"Horizon Catalog REST endpoint:\n  {HORIZON_URI}\n")
print("External engines connect using:")
print(f"  PyIceberg : RestCatalog(uri='{HORIZON_URI}', token=<session_token>)")
print(f"  Spark     : spark.sql.catalog.sf.uri = {HORIZON_URI}")
print(f"  DuckDB    : ATTACH '{HORIZON_URI}' AS h (TYPE ICEBERG_REST, ...)")
print(f"  Trino     : iceberg.rest-catalog.uri={HORIZON_URI}")

# Iceberg metadata for the table
info_raw = session.sql(f"SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('{DATABASE}.{SCHEMA}.transactions') AS i").to_pandas()['I'].iloc[0]
info = json.loads(info_raw)
print("\nIceberg table metadata:")
for k,v in info.items():
    print(f"  {k}: {str(v)[:80]}")

---
## Feature 2 — Apache Polaris Integration
Snowflake Horizon implements the same open REST protocol as Apache Polaris.
It can **serve as** a Polaris endpoint OR **read tables from** an external Polaris instance.

In [ ]:
print("Direction A — Snowflake IS the Polaris-compatible endpoint:")
print(f"  External Spark connects to: {HORIZON_URI}")
print()
print("Direction B — Snowflake reads an EXTERNAL Polaris instance:")
print("""
CREATE CATALOG INTEGRATION polaris_int
    CATALOG_SOURCE = POLARIS  TABLE_FORMAT = ICEBERG
    REST_CONFIG = (
        CATALOG_URI = 'http://<polaris-host>:8181/api/catalog'
        WAREHOUSE   = 'my_catalog'
    )
    REST_AUTHENTICATION = (
        TYPE = OAUTH
        OAUTH_TOKEN_URI     = 'http://<polaris-host>:8181/api/catalog/v1/oauth/tokens'
        OAUTH_CLIENT_ID     = '<client_id>'
        OAUTH_CLIENT_SECRET = '<client_secret>'
        OAUTH_ALLOWED_SCOPES = ('PRINCIPAL_ROLE:ALL')
    ) ENABLED = TRUE;

CREATE DATABASE polaris_db
    LINKED_CATALOG = (CATALOG_INTEGRATION = 'polaris_int')
    EXTERNAL_VOLUME = 'my_ext_vol';
""")
session.sql("SHOW INTEGRATIONS LIKE '%CATALOG%'").to_pandas()

---
## Features 3–9: Single Endpoint · Read/Write · Security · Credential Vending · Governance · Policies · Engines

In [ ]:
# Feature 3 — Single Endpoint
print("=== Feature 3: Single Endpoint ===")
tables = session.sql("SELECT table_catalog, table_schema, table_name FROM information_schema.tables WHERE table_type='ICEBERG' ORDER BY 1,2,3").to_pandas()
print(f"All Iceberg tables served from one URI ({HORIZON_URI}):")
print(tables.to_string(index=False))

# Feature 4 — Read + Write
print("\n=== Feature 4: External Engine Read + Write ===")
session.sql(f"INSERT INTO {DATABASE}.{SCHEMA}.transactions VALUES ('txn-spark','cust-X',750.00,'USD','2024-01-19 09:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west')").collect()
session.sql(f"MERGE INTO {DATABASE}.{SCHEMA}.transactions t USING (SELECT 'txn-spark' AS id,'FAILED' AS s) src ON t.transaction_id=src.id WHEN MATCHED THEN UPDATE SET t.status=src.s").collect()
session.sql(f"DELETE FROM {DATABASE}.{SCHEMA}.transactions WHERE status='FAILED'").collect()
count = session.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()['CNT'].iloc[0]
print(f"After INSERT, MERGE, DELETE: {count} rows remain")

# Feature 5 — Security Model
print("\n=== Feature 5: Security Model ===")
for sql in [
    "CREATE ROLE IF NOT EXISTS iceberg_ext_role",
    f"GRANT USAGE ON DATABASE {DATABASE} TO ROLE iceberg_ext_role",
    f"GRANT USAGE ON SCHEMA {DATABASE}.{SCHEMA} TO ROLE iceberg_ext_role",
    f"GRANT SELECT, INSERT ON TABLE {DATABASE}.{SCHEMA}.transactions TO ROLE iceberg_ext_role",
]:
    session.sql(sql).collect()
print("✓ Service role created with scoped Iceberg grants")
print(f"  Token scope for external engine: session:role:iceberg_ext_role @ {HORIZON_URI}")

In [ ]:
# Feature 6 — Credential Vending
print("=== Feature 6: Credential Vending ===")
try:
    vol_desc = session.sql(f"DESC EXTERNAL VOLUME {EXTERNAL_VOLUME}").to_pandas()
    iam_arn = vol_desc[vol_desc['property'].str.contains('IAM_USER_ARN', na=False)]['property_value'].values
    print(f"External volume IAM ARN: {iam_arn[0] if len(iam_arn) else 'N/A'}")
except Exception:
    print(f"External volume: {EXTERNAL_VOLUME} (run DESC EXTERNAL VOLUME in a worksheet to see IAM ARN)")
print("When engine sends X-Iceberg-Access-Delegation: vended-credentials, Horizon returns:")
print("  s3.access-key-id (temp STS credential, ~15 min TTL)")
print("  s3.secret-access-key  |  s3.session-token  |  s3.region")
print("Engine reads Parquet files directly from S3 — no static AWS keys needed")

# Feature 7 — Governed Multi-Engine
print("\n=== Feature 7: Governed Multi-Engine Access ===")
for role in ['spark_analyst_role','trino_analyst_role','duckdb_analyst_role']:
    session.sql(f"CREATE ROLE IF NOT EXISTS {role}").collect()
    session.sql(f"GRANT USAGE ON DATABASE {DATABASE} TO ROLE {role}").collect()
    session.sql(f"GRANT USAGE ON SCHEMA {DATABASE}.{SCHEMA} TO ROLE {role}").collect()
    session.sql(f"GRANT SELECT ON TABLE {DATABASE}.{SCHEMA}.transactions TO ROLE {role}").collect()
print("✓ spark_analyst_role, trino_analyst_role, duckdb_analyst_role — all see identical metadata")
print("  Revoking any role instantly revokes access across ALL engines")

# Feature 8 — Policy Enforcement
print("\n=== Feature 8: Policy Enforcement ===")
session.sql(f"""
CREATE OR REPLACE ROW ACCESS POLICY {DATABASE}.{SCHEMA}.rap_region AS (region_col VARCHAR)
RETURNS BOOLEAN ->
    CASE WHEN CURRENT_ROLE()='ANALYST_US' THEN region_col LIKE 'us-%'
         WHEN CURRENT_ROLE()='ANALYST_EU' THEN region_col LIKE 'eu-%'
         ELSE CURRENT_ROLE() IN ('ACCOUNTADMIN','SYSADMIN') END
""").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions ADD ROW ACCESS POLICY {DATABASE}.{SCHEMA}.rap_region ON (region)").collect()
session.sql(f"CREATE OR REPLACE MASKING POLICY {DATABASE}.{SCHEMA}.mask_customer AS (val VARCHAR) RETURNS VARCHAR -> CASE WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN','SYSADMIN') THEN val ELSE SHA2(val,256) END").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions MODIFY COLUMN customer_id SET MASKING POLICY {DATABASE}.{SCHEMA}.mask_customer").collect()
print("✓ Row access policy + masking policy applied")
print("  These policies enforce on Spark/DuckDB queries through Horizon — transparently")
try:
    display(session.sql(f"SELECT policy_name,policy_kind,ref_column_name FROM TABLE(information_schema.policy_references(ref_entity_name=>'{DATABASE}.{SCHEMA}.transactions',ref_entity_domain=>'TABLE'))").to_pandas())
except Exception:
    print("(policy_references() not available on this account — policies are applied and active)")

In [ ]:
# Feature 9 — Supported Engines reference table
print("=== Feature 9: Supported External Engines ===")
engines = pd.DataFrame([
    ('Apache Spark',  'iceberg-spark-runtime-3.5_2.12:1.7.0','Read+Write','spark.sql.catalog.sf.type=rest'),
    ('Apache Flink',  'iceberg-flink-runtime-1.18:1.7.0',    'Read+Write',"'catalog-type'='rest'"),
    ('Trino',         'Built-in (>=430)',                     'Read+Write','iceberg.rest-catalog.uri=...'),
    ('DuckDB',        'iceberg extension (>=1.1.0)',          'Read',      'ATTACH ... TYPE ICEBERG_REST'),
    ('PyIceberg',     'pyiceberg[rest]',                      'Read+Write','RestCatalog(uri=..., token=...)'),
    ('Dremio',        'Native Iceberg REST',                  'Read+Write','"type":"ICEBERG_REST"'),
    ('Apache Doris',  'Built-in (>=2.1)',                     'Read+Write',"'iceberg.catalog.type'='rest'"),
    ('StarRocks',     'Built-in (>=3.2)',                     'Read+Write','iceberg.catalog.rest.uri=...'),
], columns=['Engine','Library','DML','Key Config'])
display(engines)

---
## Features 10–18: Storage · Glue · Time Travel · Maintenance · Schema Evolution · Competitive · Streaming · Dynamic Tables · Partitioning

In [ ]:
# Feature 10 — Snowflake Storage for Iceberg
print("=== Feature 10: Snowflake Storage for Iceberg ===")
print("No S3 bucket, no IAM role, no external volume — just STORAGE_SERIALIZATION_POLICY = COMPATIBLE")
session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.orders_sf_storage (
    order_id STRING, customer_id STRING, order_date DATE,
    total_amount DECIMAL(12,2), status STRING, region STRING
)   CATALOG='SNOWFLAKE' STORAGE_SERIALIZATION_POLICY=COMPATIBLE
""").collect()
session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.orders_sf_storage VALUES
    ('ord-001','cust-A','2024-01-15',1250.00,'SHIPPED','us-west'),
    ('ord-002','cust-B','2024-01-16', 320.50,'PENDING','us-east'),
    ('ord-003','cust-C','2024-01-17',4200.00,'DELIVERED','eu-west')
""").collect()
display(session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.orders_sf_storage").to_pandas())

info = json.loads(session.sql(f"SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('{DATABASE}.{SCHEMA}.orders_sf_storage') AS i").to_pandas()['I'].iloc[0])
print(f"Still open Iceberg via Horizon: {info.get('metadataLocation','')[:60]}...")

# Feature 11 — AWS Glue + Catalog-Linked DB
print("\n=== Feature 11: AWS Glue + Catalog-Linked Databases ===")
print("""
CREATE CATALOG INTEGRATION glue_catalog_int
    CATALOG_SOURCE = ICEBERG_REST  TABLE_FORMAT = ICEBERG
    REST_CONFIG = (
        CATALOG_URI      = 'https://glue.us-east-2.amazonaws.com/iceberg'
        CATALOG_API_TYPE = AWS_GLUE   WAREHOUSE = '123456789012'
        CATALOG_NAME     = '123456789012'
        ACCESS_DELEGATION_MODE = EXTERNAL_VOLUME_CREDENTIALS
    ) ENABLED = TRUE;

CREATE DATABASE glue_linked_db
    LINKED_CATALOG = (CATALOG_INTEGRATION = 'glue_catalog_int')
    EXTERNAL_VOLUME = 'my_ext_vol';
-- Auto-discovers all Glue databases as schemas; query with lowercase + double quotes
SELECT * FROM glue_linked_db."my_glue_db"."my_table" LIMIT 10;
""")

In [ ]:
print("=== Feature 12: Iceberg Time Travel ===")
try:
    snapshots = session.sql(f"""
    SELECT snapshot_id, committed_at, operation
    FROM TABLE(information_schema.iceberg_table_history(
        TABLE_NAME=>'{DATABASE}.{SCHEMA}.transactions',
        DATABASE_NAME=>'{DATABASE.upper()}', SCHEMA_NAME=>'{SCHEMA.upper()}'
    )) ORDER BY committed_at DESC
    """).to_pandas()
    display(snapshots.head(5))
    if len(snapshots) > 0:
        snap_id = snapshots['SNAPSHOT_ID'].iloc[-1]
        print(f"\nTime travel to earliest snapshot ({snap_id}):")
        df_old = session.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE}.{SCHEMA}.transactions AT(SNAPSHOT => {snap_id})").to_pandas()
        print(f"  Row count at that snapshot: {df_old['CNT'].iloc[0]}")
except Exception as e:
    print(f"Note: iceberg_table_history not available on this account ({e})")
    print("Snapshot history is available in accounts with Horizon Catalog enabled.")
# Feature 12 — Iceberg Time Travel


print("\nTime travel 5 minutes ago:")
df_past = session.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE}.{SCHEMA}.transactions AT(TIMESTAMP => DATEADD(MINUTE,-5,CURRENT_TIMESTAMP()))").to_pandas()
print(f"  Row count: {df_past['CNT'].iloc[0]}")

# Demonstrate recovery
print("\nAccidental delete + recovery with BEFORE(STATEMENT):")
session.sql(f"DELETE FROM {DATABASE}.{SCHEMA}.transactions WHERE status='PENDING'").collect()
count_after = session.sql(f"SELECT COUNT(*) AS c FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()['C'].iloc[0]
session.sql(f"INSERT INTO {DATABASE}.{SCHEMA}.transactions SELECT * FROM {DATABASE}.{SCHEMA}.transactions BEFORE(STATEMENT=>LAST_QUERY_ID()) WHERE status='PENDING'").collect()
count_recovered = session.sql(f"SELECT COUNT(*) AS c FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()['C'].iloc[0]
print(f"  After delete: {count_after} rows  |  After recovery: {count_recovered} rows")

In [ ]:
# Feature 13 — Table Maintenance
print("=== Feature 13: Table Maintenance ===")
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions OPTIMIZE").collect()
print("✓ OPTIMIZE — compacted small Parquet files (run after bulk Spark/Flink writes)")
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions EXPIRE SNAPSHOTS OLDER_THAN = DATEADD(DAY,-7,CURRENT_TIMESTAMP())").collect()
print("✓ EXPIRE SNAPSHOTS — removed metadata older than 7 days")
print("  Tip: ALTER ICEBERG TABLE ... SET AUTO_REFRESH = TRUE for automatic background compaction")

# Feature 14 — Schema Evolution
print("\n=== Feature 14: Schema Evolution ===")
print("Before:", [r[0] for r in session.sql(f"DESCRIBE TABLE {DATABASE}.{SCHEMA}.transactions").collect()])
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions ADD COLUMN merchant_id STRING").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions RENAME COLUMN merchant_id TO merchant_ref_id").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions ADD COLUMN loyalty_pts INT DEFAULT 0").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions DROP COLUMN loyalty_pts").collect()
print("After  :", [r[0] for r in session.sql(f"DESCRIBE TABLE {DATABASE}.{SCHEMA}.transactions").collect()])
print("All column changes are metadata-only — no Parquet files rewritten")
print("External engines (Spark ALTER ICEBERG TABLE) can also evolve the schema; Snowflake picks it up automatically")

In [ ]:
# Feature 15 — Competitive Positioning
print("=== Feature 15: Competitive Positioning ===")
comp = pd.DataFrame([
    ('Snowflake + Horizon', 'Full DML',    'Native AT/BEFORE',   'RAP + Masking all engines', 'Yes',   'Open REST', 'GA'),
    ('Databricks + UC',     'Full DML',    'Delta only (native)', 'Unity Catalog only',        'Cond.', 'Open REST', 'GA w/ caveats'),
    ('AWS Glue',            'Full DML',    'Via IAM',             'Lake Formation',             'Yes',   'Open REST', 'GA'),
    ('Google BigLake',      'Read-heavy',  'Via IAM',             'BigQuery policies',          'Ltd.',  'Open REST', 'Evolving'),
    ('Microsoft Fabric',    'Delta focus', 'Via Entra ID',        'Fabric policies',            'Ltd.',  'Partial',   'Evolving'),
], columns=['Platform','DML','Time Travel','Policy Enforcement','Cred. Vending','Iceberg REST','Status'])
display(comp)
print()
print("Key Snowflake differentiators:")
for d in [
    "RAP + Masking enforced on Spark/DuckDB queries via Horizon — unique to Snowflake",
    "Native time travel (AT/BEFORE) on Iceberg tables, not just Delta",
    "OPTIMIZE, Dynamic Tables, Snowpipe Streaming all produce open Iceberg output",
    "Single governed endpoint — one URI, one RBAC system, all engines",
]:
    print(f"  ✓ {d}")

In [ ]:
# Feature 16 — Snowpipe Streaming → Iceberg
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: Snowpipe Streaming runs OUTSIDE Snowflake (Kafka Connector or Ingest SDK
# on an application server). This notebook shows:
#   PART A — Snowflake setup (runs here): create the target Iceberg table + role
#   PART B — External config (reference only): Kafka Connector properties
# ─────────────────────────────────────────────────────────────────────────────
print("=== Feature 16: Snowpipe Streaming → Iceberg ===")

# PART A: Create the Iceberg target table (runs in this notebook)
print("\n[Runs in Snowflake] Step 1 — Create target Iceberg table:")
session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.clickstream (
    event_id   STRING,
    user_id    STRING,
    event_type STRING,
    page       STRING,
    event_ts   TIMESTAMP_NTZ(6),
    region     STRING
)
    CATALOG='SNOWFLAKE'
    EXTERNAL_VOLUME='{EXTERNAL_VOLUME}'
    BASE_LOCATION='horizon_demo/clickstream/'
""").collect()
print(f"✓ Target table {DATABASE}.{SCHEMA}.clickstream created")

# PART A: Create and grant the Snowpipe Streaming role (runs in this notebook)
print("\n[Runs in Snowflake] Step 2 — Create service role for Snowpipe Streaming:")
session.sql("CREATE ROLE IF NOT EXISTS snowpipe_streaming_role").collect()
session.sql(f"GRANT USAGE ON DATABASE {DATABASE} TO ROLE snowpipe_streaming_role").collect()
session.sql(f"GRANT USAGE ON SCHEMA {DATABASE}.{SCHEMA} TO ROLE snowpipe_streaming_role").collect()
session.sql(f"GRANT INSERT ON TABLE {DATABASE}.{SCHEMA}.clickstream TO ROLE snowpipe_streaming_role").collect()
print("✓ snowpipe_streaming_role created and granted INSERT on clickstream")

# PART A: Verify table is ready and accessible via Horizon
print("\n[Runs in Snowflake] Step 3 — Verify table is open Iceberg (readable via Horizon):")
try:
    info_raw = session.sql(f"SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('{DATABASE}.{SCHEMA}.clickstream') AS i").to_pandas()['I'].iloc[0]
    import json as _json
    info = _json.loads(info_raw)
    print(f"  Metadata location: {info.get('metadataLocation','')[:80]}...")
except Exception:
    print(f"  Table {DATABASE}.{SCHEMA}.clickstream is ready for streaming ingestion")

# PART B: Kafka Connector config (reference — runs on external Kafka Connect worker)
print("""
─────────────────────────────────────────────────────────────────
[External config — Kafka Connect worker, NOT run in this notebook]

# kafka-connect-snowflake.properties
snowflake.url.name=<account>.snowflakecomputing.com
snowflake.user.name=<service_user>
snowflake.private.key=<base64_private_key>
snowflake.database.name={db}
snowflake.schema.name={schema}
snowflake.topic2table.map=clickstream_topic:{table}

# Key settings for Snowpipe Streaming → Iceberg:
snowflake.ingestion.method=SNOWPIPE_STREAMING
snowflake.output.schema.evolution=TRUE   # auto-adds new columns to Iceberg table
buffer.flush.time=10                     # seconds between flushes
buffer.count.records=10000               # rows per flush

# After each flush, data is committed as an Iceberg snapshot —
# immediately readable by external engines via Horizon REST
─────────────────────────────────────────────────────────────────
""".format(db=DATABASE, schema=SCHEMA, table='clickstream'))

# Feature 17 — Dynamic Tables as Iceberg
print("\n=== Feature 17: Dynamic Tables as Iceberg ===")
session.sql(f"""
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_daily_agg
    TARGET_LAG=DOWNSTREAM WAREHOUSE={WAREHOUSE}
    CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/txn_daily_agg/'
AS
    SELECT DATE_TRUNC('day', transaction_ts) AS txn_date, region, currency,
           COUNT(*) AS txn_count, ROUND(SUM(amount),2) AS total_amount
    FROM {DATABASE}.{SCHEMA}.transactions GROUP BY 1,2,3
""").collect()
session.sql(f"ALTER DYNAMIC TABLE {DATABASE}.{SCHEMA}.txn_daily_agg REFRESH").collect()
print("✓ DYNAMIC ICEBERG TABLE created and refreshed")
display(session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.txn_daily_agg ORDER BY txn_date").to_pandas())

# Feature 17 — Dynamic Tables as Iceberg
print("\n=== Feature 17: Dynamic Tables as Iceberg ===")
session.sql(f"""
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_daily_agg
    TARGET_LAG=DOWNSTREAM WAREHOUSE={WAREHOUSE}
    CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/txn_daily_agg/'
AS
    SELECT DATE_TRUNC('day', transaction_ts) AS txn_date, region, currency,
           COUNT(*) AS txn_count, ROUND(SUM(amount),2) AS total_amount
    FROM {DATABASE}.{SCHEMA}.transactions GROUP BY 1,2,3
""").collect()
session.sql(f"ALTER DYNAMIC TABLE {DATABASE}.{SCHEMA}.txn_daily_agg REFRESH").collect()
print("✓ DYNAMIC ICEBERG TABLE created and refreshed")
display(session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.txn_daily_agg ORDER BY txn_date").to_pandas())

In [ ]:
# Feature 18 — Partitioning + Performance Tuning + Clustering
print("=== Feature 18: Partitioning + Performance Tuning + Clustering ===")

# ── IMPORTANT: PARTITION BY vs CLUSTER BY are MUTUALLY EXCLUSIVE ─────────
# Per Snowflake docs feature matrix, clustering is ❌ unsupported on
# partitioned Iceberg tables (Snowflake-managed, CLD, or non-CLD).
# Use one or the other — NOT both on the same table.

# ── PATH A: Iceberg Partitioning (multi-engine interop) ──────────────────
# → Use when: Spark/Trino/DuckDB also read/write the table.
# → Works for ALL engines — partitioning written into Parquet/Iceberg metadata.
try:
    session.sql(f"""
    CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_partitioned (
        transaction_id STRING, customer_id STRING,
        amount DECIMAL(12,2), currency STRING,
        transaction_ts TIMESTAMP_NTZ(6), status STRING, region STRING
    )
        CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}'
        BASE_LOCATION='horizon_demo/txn_partitioned/'
        PARTITION BY ( MONTH(transaction_ts), IDENTITY(region) )
    """).collect()
    print("✓ Path A: Table partitioned by MONTH(transaction_ts) + IDENTITY(region)")
    print("  → Spark, Trino, DuckDB, Snowflake all benefit from partition pruning")
except Exception as e:
    print(f"[Info] Partition create: {e}")

try:
    session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_partitioned SET PARTITION SPEC (YEAR(transaction_ts), IDENTITY(region))").collect()
    print("✓ Partition evolution: MONTH → YEAR (zero downtime, old files unchanged)")
except Exception as e:
    print(f"[Info] Partition evolution: {e}")

try:
    session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_partitioned SET SORT ORDER (region ASC NULLS LAST, transaction_ts DESC NULLS LAST)").collect()
    print("✓ Sort order set on partitioned table (compatible with PARTITION BY)")
except Exception as e:
    print(f"[Info] Sort order: {e}")

print()
transforms = pd.DataFrame([
    ("IDENTITY(col)",    "Exact-match filter on low-cardinality string/enum columns"),
    ("MONTH(ts)",        "Date-range queries spanning weeks/months — time series data"),
    ("DAY(ts)",          "High-frequency event/log tables with daily query patterns"),
    ("YEAR(ts)",         "Long-history tables where queries span years"),
    ("BUCKET(N, col)",   "High-cardinality joins (customer_id, order_id) — hash distribute"),
    ("TRUNCATE(N, col)", "Prefix-range filters on string columns"),
], columns=["Partition Transform", "When to use"])
display(transforms)

# ── PATH B: Snowflake Clustering (Snowflake-only, non-partitioned tables) ─
# → Use when: Snowflake is the ONLY query engine AND table has no PARTITION BY.
# → Per feature matrix: clustering ❌ unsupported on partitioned Iceberg tables.
# → CLUSTER BY auto-enables Automatic Clustering — no separate enable command.
# → Uses serverless compute (no warehouse). Billed per reclustering credits.
print("\n── Path B: Clustering (Snowflake-only, non-partitioned tables) ──────")

try:
    session.sql(f"""
    CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_clustered (
        transaction_id STRING, customer_id STRING,
        amount DECIMAL(12,2), currency STRING,
        transaction_ts TIMESTAMP_NTZ(6), status STRING, region STRING
    )
        CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}'
        BASE_LOCATION='horizon_demo/txn_clustered/'
    """).collect()
    print("✓ Path B: Non-partitioned table created (no PARTITION BY)")
except Exception as e:
    print(f"[Info] Clustered table create: {e}")

try:
    # CLUSTER BY on non-partitioned table — auto-enables Automatic Clustering
    session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_clustered CLUSTER BY (region, transaction_ts)").collect()
    print("✓ CLUSTER BY set → Automatic Clustering is NOW ACTIVE (serverless)")
    print("  Control: SUSPEND RECLUSTER / RESUME RECLUSTER / DROP CLUSTERING KEY")
except Exception as e:
    print(f"[Info] CLUSTER BY: {e}")

try:
    clustering_info = session.sql(f"SELECT SYSTEM$CLUSTERING_INFORMATION('{DATABASE}.{SCHEMA}.txn_clustered')").collect()
    import json as _json
    info = _json.loads(clustering_info[0][0])
    print(f"  cluster_by_keys    : {info.get('cluster_by_keys', 'N/A')}")
    print(f"  average_overlaps   : {info.get('average_overlaps', 'N/A')}  (lower = better)")
    print(f"  average_depth      : {info.get('average_depth', 'N/A')}")
except Exception as e:
    print(f"[Info] Clustering info: {e}")

try:
    session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_clustered SUSPEND RECLUSTER").collect()
    print("✓ Clustering suspended (demo only — use RESUME RECLUSTER in production)")
except Exception as e:
    print(f"[Info] Suspend: {e}")

# ── DECISION GUIDE ────────────────────────────────────────────────────────
print()
decision = pd.DataFrame([
    ("Iceberg PARTITION BY",
     "All engines (Spark, Trino, DuckDB, Snowflake)",
     "Multi-engine workloads — interoperability is the goal",
     "❌ Cannot add CLUSTER BY to this table"),
    ("Iceberg Sort Order",
     "All engines (written into Parquet metadata)",
     "Pair with PARTITION BY for within-partition ordering",
     "✅ Compatible with PARTITION BY"),
    ("Snowflake CLUSTER BY",
     "Snowflake ONLY — invisible to Spark/Trino/DuckDB",
     "Snowflake-primary with NO PARTITION BY — micro-partition pruning",
     "❌ Not supported on partitioned Iceberg tables"),
    ("Automatic Clustering",
     "Snowflake ONLY — serverless, no warehouse",
     "Auto-enabled by CLUSTER BY — no extra command needed",
     "SUSPEND/RESUME RECLUSTER to control cost"),
], columns=["Mechanism", "Engine scope", "When to use", "Constraint"])
display(decision)

print()
print("⚠️  KEY RULE: PARTITION BY and CLUSTER BY are MUTUALLY EXCLUSIVE on Iceberg tables.")
print("   Partitioned table  → use PARTITION BY + optional Sort Order (all engines benefit)")
print("   Unpartitioned table → use CLUSTER BY if Snowflake-only (micro-partition pruning)")
print("   Docs feature matrix explicitly marks clustering ❌ for partitioned Iceberg tables.")


---
## Summary — All 18 Features Complete

**Key takeaways for customers:**

1. **One endpoint, all engines** — Horizon Catalog exposes every Iceberg table at a single REST URI
2. **True open access** — Spark, Flink, Trino, DuckDB, PyIceberg connect with zero proprietary code
3. **Governance that follows the data** — RBAC, row access policies, and masking enforce on every engine
4. **No static AWS keys** — Credential vending provides short-lived STS credentials automatically
5. **Operational excellence** — OPTIMIZE, time travel, schema evolution, Dynamic Tables all work natively
6. **Unique differentiator** — Policy enforcement on external engine queries is unique to Snowflake

**Resources:**
- Companion GitHub repo: `github.com/sfc-gh-venkatmedida/snowflake-iceberg-interop-explorer`
- Interactive Streamlit app: `AFENG_DB.ICEBERG_DEMOS.ICEBERG_INTEROP_EXPLORER` (AFE Americas account)
- Snowflake Iceberg docs: `docs.snowflake.com/en/user-guide/tables-iceberg`

---
## Features 19–28: Secure Sharing · Snowpark · Tags · Unity Catalog · PrivateLink · Iceberg v3 · WIF/OIDC · Ext Catalogs · OneLake · Auto Discovery


In [ ]:
# Feature 19 — Secure Data Sharing for Iceberg (multi-tenant)
print("=== Feature 19: Secure Data Sharing for Iceberg ===")

# Pattern A: Cross-account share
print("Pattern A — Cross-account Iceberg share:")
print("""
CREATE SHARE iceberg_data_share;
GRANT USAGE ON DATABASE horizon_demo_db        TO SHARE iceberg_data_share;
GRANT USAGE ON SCHEMA   horizon_demo_db.public  TO SHARE iceberg_data_share;
GRANT SELECT ON TABLE   horizon_demo_db.public.transactions TO SHARE iceberg_data_share;
ALTER SHARE iceberg_data_share ADD ACCOUNTS = <consumer_account_id>;
""")

# Pattern B: Multi-tenant row isolation
print("Pattern B — Multi-tenant row isolation via RAP:")
session.sql(f"""
CREATE OR REPLACE ROW ACCESS POLICY {DATABASE}.{SCHEMA}.rap_multitenant
AS (tenant_col VARCHAR) RETURNS BOOLEAN ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN','SYSADMIN') THEN TRUE
        ELSE tenant_col = SYSTEM$CURRENT_ACCOUNT()
    END
""").collect()
print("✓ Multi-tenant RAP created")
print("  Each consumer account sees ONLY its own rows — enforces in Snowflake SQL AND Horizon REST")
print(f"  Current account identity: {session.sql('SELECT SYSTEM$CURRENT_ACCOUNT()').to_pandas().iloc[0,0]}")

In [ ]:
# Feature 20 — Snowpark on Iceberg
print("=== Feature 20: Snowpark on Iceberg ===")
from snowflake.snowpark import functions as F

df = session.table(f"{DATABASE}.{SCHEMA}.transactions")
print(f"Snowpark read {df.count()} rows from Iceberg table")

agg = (df.filter(F.col("STATUS") == "COMPLETED")
         .group_by("CURRENCY")
         .agg(F.count("TRANSACTION_ID").alias("TXN_COUNT"),
              F.round(F.sum("AMOUNT"), 2).alias("TOTAL_AMOUNT"))
         .sort(F.col("TOTAL_AMOUNT").desc()))
display(agg.to_pandas())

session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.revenue_summary (
    currency STRING, txn_count BIGINT, total_amount DECIMAL(18,2)
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/revenue_summary/'
""").collect()
agg.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.revenue_summary")
print("✓ Written to Iceberg via Snowpark — open for external engines via Horizon")

# Feature 22 — Object Tags + Data Classification
print("\n=== Feature 22: Object Tags + Data Classification ===")
session.sql("CREATE TAG IF NOT EXISTS data_sensitivity ALLOWED_VALUES 'PUBLIC','INTERNAL','CONFIDENTIAL','RESTRICTED'").collect()
session.sql("CREATE TAG IF NOT EXISTS data_domain ALLOWED_VALUES 'FINANCIAL','PII','OPERATIONAL','REFERENCE'").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions SET TAG data_sensitivity='CONFIDENTIAL', data_domain='FINANCIAL'").collect()
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions MODIFY COLUMN customer_id SET TAG data_sensitivity='RESTRICTED', data_domain='PII'").collect()
print("✓ Tags applied to table and customer_id column")
try:
    tags = session.sql(f"SELECT * FROM TABLE(information_schema.tag_references('{DATABASE}.{SCHEMA}.transactions','TABLE'))").to_pandas()
        display(tags[['TAG_NAME','TAG_VALUE','DOMAIN']].head(5) if len(tags) > 0 else pd.DataFrame({'message':['Tags applied — run SHOW in Snowsight to verify']}))
except Exception:
    print('Tags applied — tag_references() not available on this account. Verify via SHOW TAG REFERENCES in Snowsight.')

In [ ]:
# Feature 22 — Unity Catalog + Snowflake Horizon: Two Distinct Patterns
print("=== Feature 22: Unity Catalog + Snowflake Horizon ===")
print("""
There are TWO separate integration patterns — do not conflate them:

─────────────────────────────────────────────────────────────────────
PATTERN A: UC Catalog Federation
  Unity Catalog federates directly to Snowflake catalog metadata.
  UC reads Snowflake-managed Iceberg files via UC external locations.

  ✅ Works with: Snowflake-MANAGED Iceberg tables only
  ❌ Does NOT use Snowflake credential vending
  📌 Storage access uses: Databricks UC external locations +
                          UC storage credentials (configured in Databricks)
  → S3/ADLS bucket must be registered as a UC external location in Databricks.

─────────────────────────────────────────────────────────────────────
PATTERN B: Spark External Catalog → Horizon REST  (UC is NOT in this path)
  Databricks Spark connects directly to Horizon as an Iceberg REST catalog.

  ✅ Uses Snowflake credential vending (vended STS/SAS tokens)
  ✅ Works on Single-User / Assigned clusters
  ⚠️  On Shared UC-enforced clusters: UC intercepts storage access —
      vended credentials are NOT honored. Use Single-User compute.

  Spark config (runs on Databricks cluster, NOT in this notebook):
    spark.sql.catalog.sf.type      = rest
    spark.sql.catalog.sf.uri       = {HORIZON_URI}
    spark.sql.catalog.sf.token     = <snowflake_service_principal_token>
    spark.sql.catalog.sf.warehouse = horizon_demo_db
    spark.sql.catalog.sf.header.X-Iceberg-Access-Delegation = vended-credentials

─────────────────────────────────────────────────────────────────────
DIRECTION B — Snowflake reads Databricks Unity Catalog (via Iceberg REST):
  CREATE CATALOG INTEGRATION unity_catalog_int
      CATALOG_SOURCE = ICEBERG_REST
      REST_CONFIG = (
          CATALOG_URI = 'https://<workspace>.azuredatabricks.net/api/2.1/unity-catalog/iceberg'
      ) ENABLED = TRUE;
  CREATE DATABASE uc_db LINKED_CATALOG = (CATALOG_INTEGRATION = 'unity_catalog_int') ...
  SELECT * FROM uc_db.main.my_table LIMIT 10;
─────────────────────────────────────────────────────────────────────
""".format(HORIZON_URI=HORIZON_URI))

try:
    display(session.sql("SHOW INTEGRATIONS LIKE '%UNITY%'").to_pandas())
except Exception:
    print("No Unity Catalog integrations configured on this account yet.")

# Feature 24 — PrivateLink
print("\n=== Feature 24: PrivateLink for Catalog Integrations ===")
print("""
Inbound PrivateLink (external engines → Horizon):
  Use: https://<account>.privatelink.snowflakecomputing.com/polaris/api/catalog

Outbound PrivateLink (Snowflake → Glue/Polaris/UC):
  CATALOG_URI = 'https://glue.us-east-2.vpce.amazonaws.com/iceberg'

Network Policy to restrict Horizon to private IPs only:
  CREATE NETWORK POLICY horizon_private ALLOWED_IP_LIST = ('10.0.0.0/8', '172.16.0.0/12');
  ALTER ACCOUNT SET NETWORK_POLICY = horizon_private;

Required for: HIPAA, PCI-DSS, FedRAMP, financial services customers
""")

In [ ]:
# Feature 24 — Iceberg v3 Features
print("=== Feature 24: Iceberg v3 Features ===")

# ── WHAT IS ACTUALLY TESTABLE IN THIS NOTEBOOK ───────────────────────────────
# ✅ Tested below: TIMESTAMP_NTZ(9), OBJECT/ARRAY columns, DEFAULT values
# ⚠️  Not directly observable from SQL (internal to Snowflake):
#     Positional deletes — Snowflake chooses delete strategy internally;
#                          cannot inspect delete file type from SQL
# ⚠️  Clarification on row lineage:
#     Iceberg v3 row lineage (_row_id) is not the same as Snowflake METADATA$ columns.
#     METADATA$ROW_ID is Snowflake-specific and exists on all Snowflake tables.
#     True Iceberg v3 row lineage tracks row origin across rewrites — not yet
#     directly queryable in Snowflake SQL as a standard v3 column.
# ─────────────────────────────────────────────────────────────────────────────

session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.events_v3 (
    event_id        STRING,
    user_id         STRING,
    event_type      STRING,
    event_ts        TIMESTAMP_NTZ(9),   -- ✅ v3: nanosecond precision
    payload         OBJECT,              -- ✅ v3: semi-structured JSON
    tags            ARRAY,               -- ✅ v3: array type
    severity        INT     DEFAULT 0,   -- ✅ v3: default column value
    is_processed    BOOLEAN DEFAULT FALSE, -- ✅ v3: default column value
    region          STRING
)   CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/events_v3/'
""").collect()

session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.events_v3 (event_id, user_id, event_type, event_ts, payload, tags, severity, region)
VALUES
    ('evt-001','usr-A','purchase','2024-01-15 09:30:00.123456789'::TIMESTAMP_NTZ(9),
     OBJECT_CONSTRUCT('product_id','P123','price',99.99),ARRAY_CONSTRUCT('high-value','first-purchase'),2,'us-west'),
    ('evt-002','usr-B','click',   '2024-01-15 09:30:00.987654321'::TIMESTAMP_NTZ(9),
     OBJECT_CONSTRUCT('page','/home','duration_ms',3200),ARRAY_CONSTRUCT('mobile'),0,'eu-west')
""").collect()

print("\n[✅ Tested] TIMESTAMP_NTZ(9), OBJECT, ARRAY, DEFAULT values:")
df_v3 = session.sql(f"""
SELECT event_id, event_ts, payload:product_id::VARCHAR AS product_id,
       payload:price::DECIMAL(10,2) AS price, tags[0]::VARCHAR AS first_tag,
       severity, is_processed   -- default FALSE was applied without explicit INSERT
FROM {DATABASE}.{SCHEMA}.events_v3
""").to_pandas()
display(df_v3)

# ── Positional deletes: execute the DELETE, note it is internal ─────────────
print("\n[✅ Runs] DELETE (positional delete — Snowflake chooses strategy internally):")
session.sql(f"DELETE FROM {DATABASE}.{SCHEMA}.events_v3 WHERE event_type = 'click'").collect()
remaining = session.sql(f"SELECT COUNT(*) AS c FROM {DATABASE}.{SCHEMA}.events_v3").to_pandas()['C'].iloc[0]
print(f"  Rows after DELETE: {remaining}")
print("  ⚠️  Whether Snowflake wrote a position-delete file or equality-delete file")
print("       is internal — not directly observable from SQL.")

# ── Row lineage: query METADATA$ pseudo-columns (Snowflake-specific) ─────────
print("\n[✅ Runs] METADATA$ pseudo-columns (Snowflake-specific, not Iceberg v3 _row_id):")
try:
    meta = session.sql(f"""
        SELECT event_id, METADATA$FILE_ROW_NUMBER AS file_row_num,
               METADATA$PARTITION_ID AS partition_id
        FROM {DATABASE}.{SCHEMA}.events_v3
    """).to_pandas()
    display(meta)
    print("  ⚠️  These are Snowflake-specific columns, not Iceberg v3 row lineage (_row_id).")
    print("       Iceberg v3 row lineage (_row_id) tracks row origin across file rewrites")
    print("       and is not yet directly queryable as a standard column in Snowflake SQL.")
except Exception as e:
    print(f"  METADATA$ columns: {e}")

print("\nEngines supporting Iceberg v3: Spark ≥3.4+Iceberg 1.5, PyIceberg ≥0.7, Trino ≥438, DuckDB ≥1.2 (read)")

In [ ]:
# Feature 25 — WIF / OIDC Authentication for External Engines
print('=== Feature 25: WIF / OIDC Authentication for External Engines ===')
print('External engines authenticate to Horizon using their native cloud identity.')
print('No static Snowflake tokens. No key rotation. GA with write-support release.')

# Concept: instead of generating a JWT and exchanging for Snowflake session token,
# external engines (EMR, Databricks, Dataproc) use their IAM role / Managed Identity
# to get an OIDC token from their cloud's identity service, then exchange that
# for a Snowflake OAuth token automatically.

# Step 1: Create security integration for AWS WIF (run as ACCOUNTADMIN)
try:
    session.sql('''
        CREATE OR REPLACE SECURITY INTEGRATION aws_wif_demo_int
            TYPE = EXTERNAL_OAUTH
            ENABLED = TRUE
            EXTERNAL_OAUTH_TYPE = CUSTOM
            EXTERNAL_OAUTH_ISSUER = 'https://sts.amazonaws.com'
            EXTERNAL_OAUTH_JWS_KEYS_URL = 'https://www.googleapis.com/oauth2/v3/certs'
            EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'sub'
            EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'login_name'
            EXTERNAL_OAUTH_AUDIENCE_LIST = ('https://scb47336.snowflakecomputing.com')
    ''').collect()
    print('[Snowflake] AWS WIF security integration created.')
except Exception as e:
    print(f'[Info] Security integration: {e}')

# Step 2: Show existing integrations
try:
    rows = session.sql("SHOW SECURITY INTEGRATIONS").collect()
    for r in rows:
        if 'WIF' in str(r).upper() or 'OAUTH' in str(r).upper() or 'EXTERNAL' in str(r).upper():
            print(f'  Integration: {r["name"]} | type: {r["type"]} | enabled: {r["enabled"]}')
except Exception as e:
    print(f'[Info] SHOW SECURITY INTEGRATIONS: {e}')

# Context: How WIF authentication flows for Spark on EMR
spark_wif_config = '''
# EMR Spark config (Iceberg REST catalog with AWS WIF — no token needed):
# spark.sql.catalog.sf                      = org.apache.iceberg.spark.SparkCatalog
# spark.sql.catalog.sf.catalog-impl         = org.apache.iceberg.rest.RESTCatalog
# spark.sql.catalog.sf.uri                  = https://scb47336.snowflakecomputing.com/polaris/api/catalog
# spark.sql.catalog.sf.warehouse            = HORIZON_DEMO_DB
# spark.sql.catalog.sf.auth-manager         = org.apache.iceberg.aws.AwsClientProperties
# spark.sql.catalog.sf.token-refresh-enabled= true
# (EMR instance profile / IAM role provides identity automatically)
'''
print('\n[Reference] Spark/EMR WIF config (no token= needed):')
print(spark_wif_config)
print('[Key insight] WIF eliminates static token management — engines use their cloud identity directly.')


In [ ]:
# Feature 26 — Supported External Catalogs (Catalog Federation reference)
print('=== Feature 26: Supported External Catalogs ===')
print('Snowflake can federate to 4 major external Iceberg REST catalogs.')
print('\nIMPORTANT: External CATALOGS (Snowflake reads FROM) vs External ENGINES (read FROM Snowflake)')
print('These are two DIFFERENT directions.\n')

# Reference table: supported external catalogs
catalogs = [
    {'Catalog': 'AWS Glue', 'CATALOG_SOURCE': 'ICEBERG_REST', 'CATALOG_API_TYPE': 'AWS_GLUE', 'Auth': 'SigV4 (IAM role)', 'Data storage': 'S3', 'Status': 'GA'},
    {'Catalog': 'Databricks Unity Catalog', 'CATALOG_SOURCE': 'ICEBERG_REST', 'CATALOG_API_TYPE': 'N/A', 'Auth': 'OAuth (service principal)', 'Data storage': 'S3/ADLS via UC ext locations', 'Status': 'GA'},
    {'Catalog': 'Apache Polaris / Open Catalog', 'CATALOG_SOURCE': 'POLARIS', 'CATALOG_API_TYPE': 'N/A', 'Auth': 'OAuth (client credentials)', 'Data storage': 'S3 (vended credentials)', 'Status': 'GA'},
    {'Catalog': 'Microsoft OneLake REST', 'CATALOG_SOURCE': 'ICEBERG_REST', 'CATALOG_API_TYPE': 'N/A', 'Auth': 'Azure OAuth (service principal)', 'Data storage': 'ADLS (OneLake)', 'Status': 'GA'},
]

print(f'{"Catalog":<30} {"CATALOG_SOURCE":<15} {"Auth":<30} {"Status"}')
print('-' * 90)
for c in catalogs:
    print(f'{c["Catalog"]:<30} {c["CATALOG_SOURCE"]:<15} {c["Auth"]:<30} {c["Status"]}')

# Show existing catalog integrations on this account
print('\n[Snowflake] Existing catalog integrations on this account:')
try:
    rows = session.sql('SHOW CATALOG INTEGRATIONS').collect()
    for r in rows:
        print(f'  {r["name"]:40} | {r["catalog_source"]:15} | enabled: {r["enabled"]}')
except Exception as e:
    print(f'  [Info] {e}')

print('\n[Key insight] Snowflake is the engine reading FROM these catalogs.')
print('[Key insight] UC auth uses UC-managed external locations — NOT Snowflake credential vending.')


In [ ]:
# Feature 27 — Microsoft OneLake REST Catalog Integration
print('=== Feature 27: Microsoft OneLake REST Catalog ===')
print('Snowflake reads OneLake-managed Iceberg tables via Iceberg REST catalog (no warehouse needed).')
print('\n⚠️  TWO PATHS for Snowflake ↔ Microsoft Fabric/OneLake:')
print('  Path A (this feature): Snowflake → OneLake via Iceberg REST  | warehouse: NOT needed')
print('  Path B (Fabric JDBC) : Fabric → Snowflake via JDBC/ODBC      | warehouse: REQUIRED')

# Show the DDL reference (not executed — requires Azure credentials in live env)
onelake_ddl = '''
-- Path A: Snowflake reads OneLake tables via Iceberg REST
CREATE OR REPLACE CATALOG INTEGRATION onelake_iceberg_int
    CATALOG_SOURCE = ICEBERG_REST
    TABLE_FORMAT   = ICEBERG
    REST_CONFIG = (
        CATALOG_URI  = 'https://api.fabric.microsoft.com/v1/workspaces/<workspace_id>/lakehouses/<lakehouse_id>/livyapi/versions/2023-12-01/spark/icebergcatalog'
        WAREHOUSE    = '<lakehouse_name>'
    )
    REST_AUTHENTICATION = (
        TYPE                 = OAUTH
        OAUTH_TOKEN_URI      = 'https://login.microsoftonline.com/<tenant_id>/oauth2/v2.0/token'
        OAUTH_CLIENT_ID      = '<azure_sp_client_id>'
        OAUTH_CLIENT_SECRET  = '<azure_sp_secret>'
        OAUTH_ALLOWED_SCOPES = ('https://analysis.windows.net/powerbi/api/.default')
    )
    ENABLED = TRUE;

-- Then create a catalog-linked database (auto-discovers OneLake lakehouses)
CREATE DATABASE onelake_db
    LINKED_CATALOG = (CATALOG_INTEGRATION = 'onelake_iceberg_int')
    EXTERNAL_VOLUME = 'onelake_ext_vol';

-- Query OneLake tables from Snowflake SQL (no warehouse usage)
SELECT * FROM onelake_db.<lakehouse>.<table> LIMIT 10;
'''
print('\n[Reference DDL — configure with real Azure credentials]:')
print(onelake_ddl)

# Private connectivity caveat
print('⚠️  PRIVATE CONNECTIVITY CAVEAT (applies to ALL catalog integrations):')
print('   Catalog-vended credentials NOT supported over PrivateLink.')
print('   Configure external volume separately for data access over private network.')
print('   Catalog integration handles metadata only over the private link.')


In [ ]:
# Feature 28 — Automatic Table Discovery + Remote Catalog Sync
print('=== Feature 28: Automatic Table Discovery + Remote Catalog Sync ===')
print('Catalog-linked databases auto-discover ALL namespaces/tables from external catalogs.')
print('No manual table registration. ALTER DATABASE ... REFRESH syncs new tables.\n')

# Verify existing CLD on this account
print('[Snowflake] Existing catalog-linked databases:')
try:
    rows = session.sql("SHOW DATABASES LIKE 'ICEBERG%'").collect()
    for r in rows:
        print(f'  {r["name"]:40} | origin: {r.get("origin", "native")}')
except Exception as e:
    print(f'  [Info] {e}')

# Demonstrate auto-discovery with existing Glue CLD
print('\n[Snowflake] Attempting to list auto-discovered tables in iceberg_glue_db:')
try:
    rows = session.sql('SHOW ICEBERG TABLES IN iceberg_glue_db').collect()
    print(f'  Auto-discovered {len(rows)} table(s):')
    for r in rows[:5]:
        print(f'    {r["database_name"]}.{r["schema_name"]}.{r["name"]}')
except Exception as e:
    print(f'  [Info] {e}')

# Sync behavior summary
print('\n[Sync Behavior]')
print('  New table added in external catalog → ALTER DATABASE <db> REFRESH → immediately queryable')
print('  Dropped table in external catalog    → disappears from Snowflake after next refresh')
print('  Schema change (column add)            → schema evolution propagated at next refresh')
print('  Auto-refresh: schedule via Snowflake Tasks for unattended sync')

# Key limitations
print('\n[CLD Limitations]')
print('  ✅ Snowflake-managed Iceberg tables')
print('  ✅ Externally-managed Iceberg tables (via catalog integration)')
print('  ❌ Regular Snowflake native tables')
print('  ❌ Delta Lake tables (unless exposed via Iceberg REST by the catalog)')
print('  ❌ Snowflake Data Sharing listings (separate path)')

print('\n=== ✅ All 28 Snowflake Iceberg Interoperability Features Covered ===')


---
## Complete Snowflake Iceberg Interoperability Story — 28 Features

| Category | Features |
|----------|---------|
| **Core interop** | 1 Open REST, 2 Horizon Catalog (Polaris-based), 3 Single Endpoint |
| **Engine access** | 4 Read+Write, 9 Engines (8 supported) |
| **Governance** | 5 Security Model, 6 Credential Vending, 7 Multi-Engine, 8 Policies, 21 Tags |
| **Storage** | 10 Snowflake Storage, 11 Catalog Federation/CLD |
| **Data management** | 12 Time Travel, 13 Automated Maintenance, 14 Schema Evolution |
| **Pipelines** | 16 Streaming, 17 Dynamic Tables |
| **Performance** | 18 Partitioning |
| **Sharing** | 19 Secure Data Sharing |
| **Native Python** | 20 Snowpark on Iceberg |
| **Multi-cloud** | 22 Unity Catalog ↔ Horizon, 23 PrivateLink |
| **Format** | 24 Iceberg v3 |
| **Security/Auth** | 25 WIF / OIDC Authentication |
| **Catalog Federation** | 26 Supported External Catalogs, 27 OneLake REST, 28 Auto Table Discovery |
| **Reference** | 15 Competitive Positioning |

**This notebook covers the complete Snowflake Iceberg interop story.**
Clone + share: `github.com/sfc-gh-venkatmedida/snowflake-iceberg-interop-explorer`
